In [14]:
import os
import sys
import asyncio
import importlib

sys.path.append(os.path.abspath(".."))
import app.ai.rag.text.normalizer as norm_module
import app.ai.rag.retrieval.retriever as ret_module

importlib.reload(norm_module)
importlib.reload(ret_module)

from app.ai.rag.text.normalizer import VietnameseNormalizer
from app.routers.dependencies import get_vector_retriever
print("✅ Đã import và cấu hình môi trường thành công!")

✅ Đã import và cấu hình môi trường thành công!


In [15]:
sample_queries = [
    "Quy định mở STK cho KHCN tại PGD",
    "Thủ tục thanh lý TSCĐ và CCLD của phòng IT",
    "Quy trình duyệt TTr mua sắm VPP trên gAMSPro",
    "Điều kiện phê duyệt HĐTD và giải ngân cho KHDN",
    "Quy định về thời hạn bàn giao BBBG và BBNT",
    "@#!#! Hello    Xin chào Anhhhhhh. khanh@ggroup.vn/"
]

print("--- Test Vietnamese Normallizer ---")
for q in sample_queries:
    normalized = VietnameseNormalizer.normalize(q, expand_acronyms=True)

    print(f"🔹 Câu gốc   : {q}")
    print(f"👉 Chuẩn hóa : {normalized}\n")

--- Test Vietnamese Normallizer ---
🔹 Câu gốc   : Quy định mở STK cho KHCN tại PGD
👉 Chuẩn hóa : Quy định mở Số tài khoản cho Khách hàng cá nhân tại Phòng giao dịch

🔹 Câu gốc   : Thủ tục thanh lý TSCĐ và CCLD của phòng IT
👉 Chuẩn hóa : Thủ tục thanh lý Tài sản cố định và Công cụ lao động của phòng Công nghệ thông tin

🔹 Câu gốc   : Quy trình duyệt TTr mua sắm VPP trên gAMSPro
👉 Chuẩn hóa : Quy trình duyệt Tờ trình mua sắm Văn phòng phẩm trên gAMSPro

🔹 Câu gốc   : Điều kiện phê duyệt HĐTD và giải ngân cho KHDN
👉 Chuẩn hóa : Điều kiện phê duyệt Hợp đồng tín dụng và giải ngân cho Khách hàng doanh nghiệp

🔹 Câu gốc   : Quy định về thời hạn bàn giao BBBG và BBNT
👉 Chuẩn hóa : Quy định về thời hạn bàn giao BG và Biên bản nghiệm thu

🔹 Câu gốc   : @#!#! Hello    Xin chào Anhhhhhh. khanh@ggroup.vn/
👉 Chuẩn hóa : Hello Xin chào Anh. khanh@ggroup.vn



In [16]:
# Khởi tạo Vector Retriever thông qua Dependency Injection
retriever = get_vector_retriever()
print("✅ VectorRetriever đã sẵn sàng:", retriever)

✅ VectorRetriever đã sẵn sàng: <app.ai.rag.retrieval.retriever.VectorRetriever object at 0x00000207B26449D0>


In [17]:
# Câu hỏi test (chứa từ viết tắt VPP, TTr, gAMSPro)
query = "Quy trình mua sắm VPP và lập TTr trên gAMSPro"

# Thực hiện truy vấn Vector Search (dùng async/await trong notebook)
results = await retriever.retrieve_context(
    query=query,
    top_k=3
)

print(f"=== 🔍 KẾT QUẢ TÌM KIẾM CHO: '{query}' ===")
docs = results.get("documents", [[]])[0]
citations = results.get("citations", [[]])[0]
distances = results.get("distances", [[]])[0]

print(f"📊 Tìm thấy: {len(docs)} đoạn tài liệu phù hợp\n")

for idx, (doc, cit) in enumerate(zip(docs, citations), 1):
    dist = distances[idx - 1] if idx - 1 < len(distances) else "N/A"
    doc_name = cit.get("document_name", "Không rõ")
    page = cit.get("page", "N/A")
    print(f"--- [Đoạn {idx}] --- (Distance: {dist})")
    print(f"📄 Nguồn: {doc_name} | Trang: {page}")
    print(f"📝 Nội dung trích dẫn:\n{doc.strip()[:300]}...\n")

=== 🔍 KẾT QUẢ TÌM KIẾM CHO: 'Quy trình mua sắm VPP và lập TTr trên gAMSPro' ===
📊 Tìm thấy: 3 đoạn tài liệu phù hợp

--- [Đoạn 1] --- (Distance: 0.4127708673477173)
📄 Nguồn: Không rõ | Trang: 30
📝 Nội dung trích dẫn:
Đ
1. PHÂN HỆ QUẢN LÝ KẾ HOẠCH
Phê duyệt tờ trình điều chuyển – GĐK, PTGĐ, TKTGĐ, TGĐ
1. PHÂN HỆ QUẢN LÝ KẾ HOẠCH
Tờ trình điều chuyển ngân sách – ĐVCM/DVDC
Bước 1: Đăng nhập hệ thống
Bước 2: Chọn mục Quản lý kế hoạch/Tờ trình chủ trương - ĐVCM/DVDC
Bước 3: Chọn loại tờ trình là Tờ trình điều chuyển
...

--- [Đoạn 2] --- (Distance: 0.43605250120162964)
📄 Nguồn: Không rõ | Trang: 24
📝 Nội dung trích dẫn:
lick nút Tìm kiếm                    để tìm.
Bước 3: Nếu có tờ trình thỏa mãn điều kiện tìm ở trên, hệ thống sẽ liệt kê các tờ trình trong danh sách (lưới) bên dưới. Click chọn tờ trình muốn duyệt trong danh sách (lưới) và click nút xem chi tiết                     . Sao đó click button             ...

--- [Đoạn 3] --- (Distance: 0.4464820623397827)
📄 Nguồn: Không rõ | Tran

In [ ]:
# Câu hỏi test (chứa từ viết tắt VPP, TTr, gAMSPro)
query = "Quy trình mua sắm VPP và lập TTr trên gAMSPro"

# Thực hiện truy vấn Vector Search (dùng async/await trong notebook)
results = await retriever.retrieve_context(
    query=query,
    top_k=3
)

print(f"=== 🔍 KẾT QUẢ TÌM KIẾM CHO: '{query}' ===")
docs = results.get("documents", [[]])[0]
citations = results.get("citations", [[]])[0]
distances = results.get("distances", [[]])[0]

print(f"📊 Tìm thấy: {len(docs)} đoạn tài liệu phù hợp\n")

for idx, (doc, cit) in enumerate(zip(docs, citations), 1):
    dist = distances[idx - 1] if idx - 1 < len(distances) else "N/A"
    doc_name = cit.get("document_name", "Không rõ")
    page = cit.get("page", "N/A")
    print(f"--- [Đoạn {idx}] --- (Distance: {dist})")
    print(f"📄 Nguồn: {doc_name} | Trang: {page}")
    print(f"📝 Nội dung trích dẫn:\n{doc.strip()[:300]}...\n")

=== 🔍 KẾT QUẢ TÌM KIẾM CHO: 'Quy trình mua sắm VPP và lập TTr trên gAMSPro' ===
📊 Tìm thấy: 3 đoạn tài liệu phù hợp

--- [Đoạn 1] --- (Distance: 0.4127708673477173)
📄 Nguồn: Không rõ | Trang: 30
📝 Nội dung trích dẫn:
Đ
1. PHÂN HỆ QUẢN LÝ KẾ HOẠCH
Phê duyệt tờ trình điều chuyển – GĐK, PTGĐ, TKTGĐ, TGĐ
1. PHÂN HỆ QUẢN LÝ KẾ HOẠCH
Tờ trình điều chuyển ngân sách – ĐVCM/DVDC
Bước 1: Đăng nhập hệ thống
Bước 2: Chọn mục Quản lý kế hoạch/Tờ trình chủ trương - ĐVCM/DVDC
Bước 3: Chọn loại tờ trình là Tờ trình điều chuyển
...

--- [Đoạn 2] --- (Distance: 0.43605250120162964)
📄 Nguồn: Không rõ | Trang: 24
📝 Nội dung trích dẫn:
lick nút Tìm kiếm                    để tìm.
Bước 3: Nếu có tờ trình thỏa mãn điều kiện tìm ở trên, hệ thống sẽ liệt kê các tờ trình trong danh sách (lưới) bên dưới. Click chọn tờ trình muốn duyệt trong danh sách (lưới) và click nút xem chi tiết                     . Sao đó click button             ...

--- [Đoạn 3] --- (Distance: 0.4464820623397827)
📄 Nguồn: Không rõ | Tran